In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [5]:
# Load
df = pd.read_csv('/content/drive/MyDrive/5G_NIDD_multiclass_clean.csv', low_memory=False)

print("Original shape:", df.shape)

# Target
y = df['Label']
X = df.drop(columns=['Label', 'Attack Type', 'Attack Tool'], errors='ignore')

# Remove obvious non-learning columns
drop_cols = [
    'SrcMac','DstMac','SrcAddr','DstAddr','StartTime','LastTime',
    'SrcOui','DstOui'
]

X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')

# Keep only numeric features
X = X.select_dtypes(include=[np.number])

print("After numeric selection:", X.shape)


Original shape: (1215890, 112)
After numeric selection: (1215890, 86)


In [6]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)


In [7]:
selector = SelectKBest(score_func=f_classif, k=36)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected Features:", selected_features.tolist())


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 1  7 13 14 19 20 21 22 23 24 25 26 27 34 35 36 37 48 49 50 51 57 58 59
 60 61 62 63 64 65 66 67 68 69 70 71 72 77 78 79 80] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Selected Features: ['Rank', 'Seq', 'Dur', 'RunTime', 'Mean', 'Sum', 'Min', 'Max', 'sTos', 'dTos', 'sTtl', 'dTtl', 'sHops', 'dHops', 'TotPkts', 'SrcPkts', 'DstPkts', 'TotBytes', 'SrcBytes', 'DstBytes', 'Offset', 'sMeanPktSz', 'dMeanPktSz', 'Loss', 'SrcLoss', 'DstLoss', 'pLoss', 'SrcWin', 'DstWin', 'sVid', 'dVid', 'SrcTCPBase', 'DstTCPBase', 'TcpRtt', 'SynAck', 'AckDat']


In [8]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_classes = len(np.unique(y_encoded))
print("Classes:", num_classes)


Classes: 20


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)


In [10]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)   # FIT ONLY TRAIN
X_test  = scaler.transform(X_test)        # TRANSFORM TEST


In [11]:
X_train = X_train.reshape(-1, 36, 1)
X_test  = X_test.reshape(-1, 36, 1)


In [12]:
def MobileNetV1_BiGRU(drop_rate=0.5, gru_units=128, dense_units=256):

    inp = Input(shape=(36,1))

    # ----- CNN BRANCH -----
    x = Reshape((36,1,1))(inp)

    x = Conv2D(32,(3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(64,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(128,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    cnn_out = GlobalAveragePooling2D()(x)

    # ----- GRU BRANCH -----
    r = Bidirectional(GRU(gru_units, return_sequences=True))(inp)
    r = Bidirectional(GRU(gru_units))(r)

    # ----- FUSION -----
    merged = Concatenate()([cnn_out, r])

    merged = Dense(dense_units, activation="relu")(merged)
    merged = Dense(dense_units//2, activation="relu")(merged)
    merged = Dropout(drop_rate)(merged)

    out = Dense(num_classes, activation="softmax")(merged)

    model = Model(inp, out)
    return model


In [13]:
!pip install keras-tuner


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 6.5 MB/s eta 0:00:00


In [14]:
import tensorflow as tf

def focal_loss(gamma=2.0, alpha=0.25):

    def loss(y_true, y_pred):

        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.one_hot(y_true, depth=num_classes)

        ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)

        pt = tf.exp(-ce)
        focal = alpha * tf.pow(1 - pt, gamma) * ce

        return tf.reduce_mean(focal)

    return loss


In [15]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights))
print(class_weights)


{np.int64(0): np.float64(0.64811172410117), np.int64(1): np.float64(0.6491671115856914), np.int64(2): np.float64(3.325965944060726), np.int64(3): np.float64(4.20650406504065), np.int64(4): np.float64(37.819284603421465), np.int64(5): np.float64(44.74296228150874), np.int64(6): np.float64(1.3619983757596124), np.int64(7): np.float64(4.309374446216552), np.int64(8): np.float64(5.309563318777292), np.int64(9): np.float64(5.274438781043271), np.int64(10): np.float64(1.9601644365629534), np.int64(11): np.float64(4.803516049382716), np.int64(12): np.float64(5.220652640618291), np.int64(13): np.float64(5.217292426517915), np.int64(14): np.float64(1.5948189926547744), np.int64(15): np.float64(1.9197000197355436), np.int64(16): np.float64(0.12998123867505493), np.int64(17): np.float64(0.21242149215139894), np.int64(18): np.float64(6.053721682847897), np.int64(19): np.float64(5.8995147986414365)}


In [ ]:
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping

def build_model(hp):

    model = MobileNetV1_BiGRU(
        drop_rate = hp.Choice("dropout",[0.3,0.5,0.6]),
        gru_units = hp.Choice("gru_units",[64,128,256]),
        dense_units = hp.Choice("dense",[128,256,512])
    )

    lr = hp.Choice("lr",[1e-2,1e-3,1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=2, alpha=0.25),
        metrics=["accuracy"]
    )

    return model


tuner = kt.Hyperband(
    build_model,
    objective="val_accuracy",
    max_epochs=7,   # normally 10
    factor=3,
    directory="tuning",
    project_name="5g_mobilenet_bigru"
)

# ---- Tune ONLY on subset ----
sample_idx = np.random.choice(len(X_train), size=int(len(X_train)*0.25), replace=False)
X_tune = X_train[sample_idx]
y_tune = y_train[sample_idx]

stop_early = EarlyStopping(monitor='val_loss', patience=3)

tuner.search(
    X_tune, y_tune,
    validation_split=0.2,
    epochs=10,
    batch_size=512,
    callbacks=[stop_early],
    verbose=1
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best Hyperparameters:")
print(best_hps.values)


Trial 10 Complete [00h 04m 36s]
val_accuracy: 0.8654905557632446

Best val_accuracy So Far: 0.8736121654510498
Total elapsed time: 00h 23m 53s
Best Hyperparameters:
{'dropout': 0.3, 'gru_units': 128, 'dense': 256, 'lr': 0.001, 'tuner/epochs': 7, 'tuner/initial_epoch': 3, 'tuner/bracket': 1, 'tuner/round': 1, 'tuner/trial_id': '0001'}


In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=512,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=8, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)


Epoch 1/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 82s 45ms/step - accuracy: 0.7106 - loss: 0.7344 - val_accuracy: 0.8587 - val_loss: 0.3191 - learning_rate: 0.0010
Epoch 2/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 78s 46ms/step - accuracy: 0.8538 - loss: 0.3301 - val_accuracy: 0.8432 - val_loss: 0.4126 - learning_rate: 0.0010
Epoch 3/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8636 - loss: 0.3027 - val_accuracy: 0.8264 - val_loss: 0.6040 - learning_rate: 0.0010
Epoch 4/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 77s 45ms/step - accuracy: 0.8723 - loss: 0.2777 - val_accuracy: 0.8607 - val_loss: 0.3999 - learning_rate: 0.0010
Epoch 5/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 45ms/step - accuracy: 0.8778 - loss: 0.2630 - val_accuracy: 0.8354 - val_loss: 0.4509 - learning_rate: 0.0010
Epoch 6/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8923 - loss: 0.2365 - val_accuracy: 0.8893 - val_loss: 0.2465 - learning_rate: 1.0000e-04
Epoch 7/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - ac

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=512,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=8, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)


Epoch 1/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 91s 44ms/step - accuracy: 0.7066 - loss: 0.1117 - val_accuracy: 0.7704 - val_loss: 0.0933 - learning_rate: 0.0010
Epoch 2/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 45ms/step - accuracy: 0.8510 - loss: 0.0416 - val_accuracy: 0.8657 - val_loss: 0.0355 - learning_rate: 0.0010
Epoch 3/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8694 - loss: 0.0348 - val_accuracy: 0.8698 - val_loss: 0.0345 - learning_rate: 0.0010
Epoch 4/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8761 - loss: 0.0313 - val_accuracy: 0.8441 - val_loss: 0.0650 - learning_rate: 0.0010
Epoch 5/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8826 - loss: 0.0291 - val_accuracy: 0.8768 - val_loss: 0.0347 - learning_rate: 0.0010
Epoch 6/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8862 - loss: 0.0276 - val_accuracy: 0.8909 - val_loss: 0.0250 - learning_rate: 0.0010
Epoch 7/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accura

In [ ]:
pred = model.predict(X_test)
pred = np.argmax(pred, axis=1)

print(classification_report(y_test, pred))


7600/7600 ━━━━━━━━━━━━━━━━━━━━ 38s 5ms/step
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     18761
           1       0.94      0.96      0.95     18730
           2       0.57      0.77      0.66      3656
           3       0.58      0.89      0.70      2890
           4       0.39      0.52      0.44       322
           5       0.34      0.16      0.22       271
           6       0.98      0.81      0.89      8927
           7       0.86      0.62      0.72      2822
           8       0.90      0.91      0.90      2290
           9       0.99      0.87      0.93      2305
          10       0.84      0.85      0.85      6203
          11       0.53      0.74      0.62      2531
          12       0.53      0.13      0.21      2329
          13       0.52      0.83      0.64      2331
          14       0.95      0.95      0.95      7624
          15       0.99      0.96      0.98      6334
          16       0.97      0.91    

In [ ]:
pred = model.predict(X_test)
pred = np.argmax(pred, axis=1)

print(classification_report(y_test, pred))


7600/7600 ━━━━━━━━━━━━━━━━━━━━ 38s 5ms/step
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     18761
           1       0.97      0.97      0.97     18730
           2       0.68      0.92      0.78      3656
           3       0.84      0.78      0.81      2890
           4       0.33      0.65      0.43       322
           5       0.32      0.27      0.29       271
           6       0.98      0.94      0.96      8927
           7       0.94      0.93      0.94      2822
           8       0.94      0.89      0.92      2290
           9       0.94      0.88      0.91      2305
          10       0.82      0.81      0.82      6203
          11       0.57      0.58      0.58      2531
          12       0.58      0.08      0.14      2329
          13       0.51      0.88      0.64      2331
          14       0.97      0.94      0.95      7624
          15       0.98      0.96      0.97      6334
          16       0.99      0.86    

In [ ]:
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping

def build_model(hp):

    model = MobileNetV1_BiGRU(
        drop_rate = hp.Float(
            "dropout",
            min_value=0.2,
            max_value=0.6,
            step=0.1
        ),

        gru_units = hp.Int(
            "gru_units",
            min_value=64,
            max_value=256,
            step=32
        ),

        dense_units = hp.Int(
            "dense",
            min_value=128,
            max_value=512,
            step=64
        )
    )

    lr = hp.Float(
        "lr",
        min_value=1e-5,
        max_value=1e-2,
        sampling="log"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=2, alpha=0.25),
        metrics=["accuracy"]
    )

    return model


tuner = kt.Hyperband(
    build_model,
    objective="val_accuracy",
    max_epochs=7,
    factor=3,
    directory="tuning",
    project_name="5g_mobilenet_bigru"
)

# ---- Tune ONLY on subset ----
sample_idx = np.random.choice(len(X_train), size=int(len(X_train)*0.75), replace=False)
X_tune = X_train[sample_idx]
y_tune = y_train[sample_idx]

stop_early = EarlyStopping(monitor='val_loss', patience=3)

tuner.search(
    X_tune,
    y_tune,
    validation_split=0.2,
    epochs=10,
    batch_size=512,
    callbacks=[stop_early],
    verbose=1
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best Hyperparameters:")
print(best_hps.values)

Trial 4 Complete [00h 02m 50s]
val_accuracy: 0.8122296929359436

Best val_accuracy So Far: 0.8706847429275513
Total elapsed time: 00h 14m 57s

Search: Running Trial #5

Value             |Best Value So Far |Hyperparameter
0.2               |0.2               |dropout
224               |160               |gru_units
320               |384               |dense
1.927e-05         |0.00058335        |lr
3                 |3                 |tuner/epochs
0                 |0                 |tuner/initial_epoch
1                 |1                 |tuner/bracket
0                 |0                 |tuner/round

Epoch 1/3
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 110s 91ms/step - accuracy: 0.4626 - loss: 0.3707 - val_accuracy: 0.6535 - val_loss: 0.1407
Epoch 2/3
1140/1140 ━━━━━━━━━━━━━━━━━━━━ 103s 91ms/step - accuracy: 0.6287 - loss: 0.1394 - val_accuracy: 0.6867 - val_loss: 0.1091
Epoch 3/3
 318/1140 ━━━━━━━━━━━━━━━━━━━━ 1:08 83ms/step - accuracy: 0.6746 - loss: 0.1132

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=512,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=8, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)


In [ ]:
pred = model.predict(X_test)
pred = np.argmax(pred, axis=1)

print(classification_report(y_test, pred))
